# 🔵 Lesson 3.4 — Clustering: Finding Hidden Groups

**AI Crash Course for Absolute Beginners**

Clustering is **unsupervised** — the data has no labels, and we let the algorithm discover structure. In this notebook:
- Run K-Means and watch it converge step by step
- Use the Elbow method to choose K automatically
- Try DBSCAN for irregular shapes
- Apply clustering to real customer segmentation

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN         # clustering algorithms
from sklearn.datasets import make_blobs, make_moons # tools to generate test datasets
from sklearn.preprocessing import StandardScaler    # scale features before clustering
from sklearn.metrics import silhouette_score        # measures how well-separated clusters are

---
## Part 1 — K-Means Step by Step

In [ ]:
# Generate 3 natural clusters of data points for demonstration
# make_blobs creates circular blobs — perfect for showing K-Means
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

COLORS = ["#c8401a", "#1a6bc8", "#2a8c5a", "#b5820a", "#8c1a5a"]

# Show K-Means at 3 different stages to visualise how it converges
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, n_iter in enumerate([1, 3, 10]):
    # max_iter limits how many steps the algorithm takes (so we can see intermediate stages)
    # n_init=1 means only one random starting point (for consistent visualisation)
    # init="random" means centroids start at random positions
    km = KMeans(n_clusters=3, max_iter=n_iter, n_init=1,
                init="random", random_state=0)

    # fit_predict trains the model AND returns cluster labels in one step
    labels  = km.fit_predict(X)
    centers = km.cluster_centers_   # the centroid (centre point) of each cluster

    for c in range(3):
        mask = labels == c          # True for all points in cluster c
        axes[i].scatter(X[mask, 0], X[mask, 1], color=COLORS[c], alpha=0.5, s=20)

    # Plot centroids as large X markers
    axes[i].scatter(centers[:, 0], centers[:, 1], s=200, color="black",
                    marker="x", linewidths=3, label="Centroids")
    axes[i].set_title(f"After {n_iter} iteration{'s' if n_iter>1 else ''}")
    axes[i].axis("equal")

plt.suptitle("K-Means Convergence", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2 — Choosing K: The Elbow Method

In [ ]:
inertia_list = []   # inertia = total distance from each point to its centroid (lower = tighter clusters)
sil_list     = []   # silhouette score = how well-separated the clusters are (higher = better, max 1.0)
K_range      = range(2, 11)   # test K values from 2 to 10

for k in K_range:
    # n_init=10 means run K-Means 10 times with different starting points and keep the best result
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertia_list.append(km.inertia_)              # total within-cluster variance
    sil_list.append(silhouette_score(X, labels))  # average cluster separation score

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Elbow method: look for the "elbow" where inertia stops dropping sharply
axes[0].plot(K_range, inertia_list, marker="o", color="#1a6bc8")
axes[0].axvline(3, color="#c8401a", linestyle="--", label="Elbow at k=3")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia (within-cluster variance)")
axes[0].set_title("Elbow Method")
axes[0].legend()

# Silhouette: the K with the highest score gives the best-separated clusters
axes[1].plot(K_range, sil_list, marker="o", color="#2a8c5a")
axes[1].axvline(3, color="#c8401a", linestyle="--", label="Best k=3")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score (higher = better)")
axes[1].set_title("Silhouette Analysis")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Best k by silhouette: {list(K_range)[sil_list.index(max(sil_list))]}")

---
## Part 3 — DBSCAN: Handles Irregular Shapes

In [ ]:
# K-Means assumes round/circular clusters — it fails on crescent or ring shapes
# DBSCAN (Density-Based Spatial Clustering) finds clusters of any shape

X_moon, _ = make_moons(n_samples=300, noise=0.08, random_state=42)   # two crescent shapes

km_moon = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moon)

# DBSCAN parameters:
# eps = how close two points must be to be considered neighbours (smaller = tighter clusters)
# min_samples = minimum number of points to form a cluster (higher = stricter)
db_moon = DBSCAN(eps=0.2, min_samples=5).fit(X_moon)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# K-Means result (will get it wrong — can't separate crescents)
for label, color in zip([0, 1], [COLORS[0], COLORS[1]]):
    mask = km_moon.labels_ == label
    axes[0].scatter(X_moon[mask, 0], X_moon[mask, 1], color=color, alpha=0.6, s=20)
axes[0].set_title("K-Means (fails on crescents)")

# DBSCAN result — label=-1 means the point is "noise" (doesn't belong to any cluster)
unique_labels = set(db_moon.labels_)
for label in unique_labels:
    mask  = db_moon.labels_ == label
    color = "grey" if label == -1 else COLORS[label]   # grey for noise points
    axes[1].scatter(X_moon[mask, 0], X_moon[mask, 1], color=color, alpha=0.6, s=20,
                    label="noise" if label == -1 else f"cluster {label}")
axes[1].set_title("DBSCAN (handles crescents correctly)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 4 — Real Application: Customer Segmentation

In [ ]:
# Simulate customer purchase data
np.random.seed(0)
customers = pd.DataFrame({
    "age"              : np.random.randint(18, 70, 500),
    "annual_income_k"  : np.random.normal(60, 25, 500).clip(15, 200),
    "spending_score"   : np.random.randint(1, 101, 500),
    "purchase_freq"    : np.random.poisson(6, 500),
})

print(customers.describe().round(1))

In [ ]:
# Scale and cluster into 4 segments
scaler = StandardScaler()
X_cust = scaler.fit_transform(customers)

km = KMeans(n_clusters=4, n_init=15, random_state=42)
customers["segment"] = km.fit_predict(X_cust)

print("Cluster sizes:")
print(customers["segment"].value_counts().sort_index())

In [ ]:
# Profile each segment
profiles = customers.groupby("segment").mean().round(1)

labels_map = {
    profiles["spending_score"].idxmax()  : "High Spenders",
    profiles["spending_score"].idxmin()  : "Low Spenders",
    profiles["annual_income_k"].idxmax() : "High Income",
    profiles["annual_income_k"].idxmin() : "Budget Shoppers"
}

profiles["Label"] = profiles.index.map(labels_map).fillna("Mixed")
print(profiles)

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(8, 5))
for seg in range(4):
    subset = customers[customers["segment"] == seg]
    ax.scatter(subset["annual_income_k"], subset["spending_score"],
               label=labels_map.get(seg, f"Segment {seg}"),
               color=COLORS[seg], alpha=0.6, s=30)

ax.set_xlabel("Annual Income ($k)")
ax.set_ylabel("Spending Score (1-100)")
ax.set_title("Customer Segments")
ax.legend()
plt.tight_layout()
plt.show()

---
## Challenge Exercises

1. **PCA visualisation**: Reduce the 4-feature customer dataset to 2D using `sklearn.decomposition.PCA` and re-plot the clusters.
2. **Hierarchical clustering**: Try `from sklearn.cluster import AgglomerativeClustering` and plot a dendrogram with `scipy.cluster.hierarchy`.
3. **DBSCAN tuning**: Try different `eps` values (0.1, 0.3, 0.5) on the moons dataset and see how noise points change.

---
**Next lesson:** [Lesson 3.5 — Overfitting and Generalisation](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-3.5-overfitting.ipynb)